# NB-P02-02 — Calculus and Optimisation

Companion notebook for [LP02-04](../lessons/LP02-04-derivatives-partial-derivatives-and-gradients.md)
and [LP02-05](../lessons/LP02-05-optimization-fundamentals.md).

The rule for this notebook: **never trust a derivative you have not checked numerically.** The
`numeric_grad` function below is the tool for that, and you will use it in M03 and beyond.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("NumPy", np.__version__)

## 1. A numerical gradient you can reuse

Central differences: $f'(x) \approx \frac{f(x+h) - f(x-h)}{2h}$, with $h \approx 10^{-5}$.

In [ ]:
def numeric_grad(f, x, h=1e-5):
    """Central-difference gradient of a scalar function of a vector."""
    x = np.atleast_1d(np.asarray(x, dtype=float))
    grad = np.zeros_like(x)
    for i in range(x.size):
        step = np.zeros_like(x)
        step[i] = h
        grad[i] = (f(x + step) - f(x - step)) / (2 * h)
    return grad


print("d/dx x^2 at 3 :", numeric_grad(lambda v: v[0] ** 2, [3.0]))
print("analytic      : 6.0")

### Why not make $h$ tiny?

Because $f(x+h)$ and $f(x-h)$ become nearly equal, and subtracting two nearly equal floats destroys
precision. There is an optimum, and it is around $10^{-5}$ for double precision.

In [ ]:
f = lambda v: v[0] ** 2
true = 6.0
for h in (1e-1, 1e-3, 1e-5, 1e-7, 1e-9, 1e-11, 1e-13):
    est = float((f([3.0 + h]) - f([3.0 - h])) / (2 * h))
    print(f"h={h:>7.0e}  estimate={est:>18.10f}  abs error={abs(est - true):.2e}")

## 2. Partial derivatives and the gradient

Hold every other variable constant, differentiate with respect to one.

In [ ]:
def g(v):
    x, y = v
    return x**2 * y + 3 * y


point = np.array([2.0, 1.0])
analytic = np.array([2 * point[0] * point[1], point[0] ** 2 + 3])
print("analytic:", analytic)
print("numeric :", numeric_grad(g, point).round(6))
print("match   :", np.allclose(analytic, numeric_grad(g, point)))

## 3. The chain rule

$h(x) = (3x+1)^2$, so $h'(x) = 6(3x+1)$. Expanding first gives $9x^2 + 6x + 1$, whose derivative
$18x + 6$ agrees — a cross-check worth doing by hand.

In [ ]:
h = lambda v: (3 * v[0] + 1) ** 2
chain = lambda x: 6 * (3 * x + 1)
expanded = lambda x: 18 * x + 6

for x in (0.0, 1.0, -2.5):
    print(f"x={x:>5}: chain={chain(x):>7.2f} expanded={expanded(x):>7.2f} numeric={numeric_grad(h, [x])[0]:.6f}")

## 4. Gradient descent on a quadratic

$f(x) = x^2$, $f'(x) = 2x$, update $x \leftarrow x - \eta\,2x$. With $\eta = 0.25$ each step halves the
distance to zero.

In [ ]:
def descend(f_prime, x0, lr, steps=8):
    x = float(x0)
    path = [x]
    for _ in range(steps):
        x = x - lr * f_prime(x)
        path.append(x)
    return path


print("lr=0.25:", [round(v, 4) for v in descend(lambda x: 2 * x, 4.0, 0.25, 5)])
print("lr=0.9 :", [round(v, 4) for v in descend(lambda x: 2 * x, 4.0, 0.9, 5)])
print("lr=1.5 :", [round(v, 2) for v in descend(lambda x: 2 * x, 4.0, 1.5, 5)])

The third case diverges because $\eta = 1.5$ exceeds $2/L = 1$ for curvature $L = 2$. The second
converges but oscillates: $1 - 2\eta = -0.8$, so the sign flips every step.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.2))
for lr, style in [(0.1, "-o"), (0.25, "-s"), (0.9, "-^"), (1.5, "-x")]:
    path = descend(lambda x: 2 * x, 4.0, lr, 8)
    ax.semilogy([abs(v) + 1e-12 for v in path], style, label=f"lr={lr}")
ax.set_xlabel("step")
ax.set_ylabel("|x| (log scale)")
ax.set_title("Learning rate decides convergence, oscillation or divergence")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Two dimensions: a path on the loss surface

$f(x, y) = x^2 + 3y^2$, $\nabla f = (2x, 6y)$. The $y$ direction is three times steeper, so one shared
learning rate suits them differently.

In [ ]:
def f2(p):
    return p[0] ** 2 + 3 * p[1] ** 2


def grad2(p):
    return np.array([2 * p[0], 6 * p[1]])


def run(p0, lr, steps=25):
    p = np.array(p0, dtype=float)
    path = [p.copy()]
    for _ in range(steps):
        p = p - lr * grad2(p)
        path.append(p.copy())
    return np.array(path)


path = run([2.0, 1.0], 0.1)
print("final point:", path[-1].round(6))
print("loss at start / end:", round(float(f2(path[0])), 4), "->", f"{f2(path[-1]):.2e}")

xs = np.linspace(-2.5, 2.5, 200)
ys = np.linspace(-1.5, 1.5, 200)
XX, YY = np.meshgrid(xs, ys)
ZZ = XX**2 + 3 * YY**2

fig, ax = plt.subplots(figsize=(5, 3.4))
cs = ax.contour(XX, YY, ZZ, levels=12, cmap="viridis")
ax.plot(path[:, 0], path[:, 1], "-o", color="crimson", markersize=3, label="descent path")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Gradient descent crosses the contours")
ax.legend()
fig.colorbar(cs, ax=ax, label="f(x, y)")
plt.tight_layout()
plt.show()

## 6. Gradients of a loss function

For $L(w,b) = \frac{1}{n}\sum (wx_i + b - y_i)^2$:

$$\frac{\partial L}{\partial w} = \frac{2}{n}\sum x_i(\hat{y}_i - y_i), \qquad
\frac{\partial L}{\partial b} = \frac{2}{n}\sum (\hat{y}_i - y_i)$$

In [ ]:
X = np.array([1.0, 2.0, 3.0, 4.0])
Y = np.array([3.0, 5.0, 7.0, 8.5])


def loss(p):
    w, b = p
    return float(np.mean((w * X + b - Y) ** 2))


def grad_loss(p):
    w, b = p
    resid = w * X + b - Y
    return np.array([2 * np.mean(X * resid), 2 * np.mean(resid)])


p = np.array([1.0, 0.0])
print("analytic:", grad_loss(p).round(4))
print("numeric :", numeric_grad(loss, p).round(4))

for step in range(4):
    p = p - 0.1 * grad_loss(p)
    print(f"step {step + 1}: w={p[0]:.3f} b={p[1]:.3f} loss={loss(p):.4f}")

Compare with `np.linalg.lstsq` on the same design matrix — after enough steps the two must agree.

In [ ]:
design = np.column_stack([X, np.ones_like(X)])
print("lstsq:", np.linalg.lstsq(design, Y, rcond=None)[0].round(3))

p = np.array([1.0, 0.0])
for _ in range(2000):
    p = p - 0.05 * grad_loss(p)
print("descent:", p.round(3))

## 7. Exercises

1. Verify numerically that $\nabla_{\mathbf{x}}(\mathbf{x}^\top A\mathbf{x}) = (A + A^\top)\mathbf{x}$
   for a random 3×3 matrix.
2. For $f(x) = x^4 - 3x^2$, find all stationary points analytically and classify each with the second
   derivative. Then run descent from $x_0 = 0.0$, from $x_0 = 0.05$ and from $x_0 = -0.05$, and explain
   why the first run does not move.
3. Run the 2-D descent with $\eta = 0.5$ and describe what happens to each coordinate.

In [ ]:
# 1
rng = np.random.default_rng(0)
A = rng.normal(size=(3, 3))
x = rng.normal(size=3)
analytic = (A + A.T) @ x
numeric = numeric_grad(lambda v: v @ A @ v, x)
print(analytic.round(6))
print(numeric.round(6))
print(np.allclose(analytic, numeric))

In [ ]:
# 2
stat = lambda x: 4 * x**3 - 6 * x
curve = lambda x: x**4 - 3 * x**2
print("stationary: 0, +-sqrt(1.5); f'' = 12x^2 - 6, so 0 is a maximum and +-1.2247 are minima")
for x0 in (0.0, 0.05, -0.05):
    x = x0
    for _ in range(200):
        x -= 0.05 * stat(x)
    print(f"start {x0:>5}: converged to {x:>8.4f} (f={curve(x):.4f})")

In [ ]:
# 3
path = run([2.0, 1.0], 0.5, steps=6)
for i, point in enumerate(path):
    print(i, point.round(6))

## 8. Check yourself

1. Why does the numerical gradient get *worse* as $h$ becomes smaller than about $10^{-8}$?
2. What does a learning rate above $2/L$ do, and why is $L$ the relevant quantity?
3. Why is a saddle point harder for gradient descent than a local minimum?

Next: [P03 — Probability and Statistics](../../P03-probability-and-statistics/README.md).